# 第2回 宿題（発展）: 講座の用語集で LoRA 学習し、学習前後と検索を比べる

演習のノートブック（`notebooks/lora_format_tuning.ipynb`）と同じモデル、同じ版、同じ LoRA の設定で、架空 FAQ の代わりに講座の用語集（58 語、116 件の Q&A）を学習させます。

**演習との違い**

| 項目 | 演習 | この宿題 |
| --- | --- | --- |
| 学習データ | 架空 FAQ の 14 件（JSON で答える） | 用語集の 116 件（説明文で答える。`scripts/make_sft_dataset.py` の出力） |
| 入力 | システムプロンプト＋FAQ＋質問 | 質問だけ（学習データと同じ形） |
| 周回 | 20 エポック（40 ステップ） | 10 エポック（150 ステップ）。1 語に質問が 2 通りあるので、同じ説明文を 20 回見る |
| 答えの長さの上限 | 96 トークン | 256 トークン（説明文は最長 212 字） |
| 評価 | 21 問を形式と内容で採点 | 25 問を 3 区分で採点（学習と同じ言い回し 10、言い回しを変えた質問 10、用語集にない語 5） |
| 保存先 | `MyDrive/aie2026/qlora_lab` | `MyDrive/aie2026/homework_glossary`（演習の結果を上書きしない） |

**使い方**: ランタイムのタイプを T4 GPU にしてから、セルを上から 1 つずつ実行します。学習データ（`glossary_sft.jsonl`）と評価の質問（`eval_questions.json`）は、Drive の保存先に置いてあるものを読みます。

## 0. 環境の確認とライブラリの導入

演習と同じです。1 つ目のセルで `Tesla T4` と出ること、2 つ目のセルの最後に版の一覧が出ることを確かめます。赤い `ERROR: pip's dependency resolver ...`（cudf や pyarrow）は演習と同じく無視してよい表示です。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイムのタイプを T4 GPU にしてください。")

In [ ]:
# 演習のノートブックと同じ版に固定する（Colab の T4 で動作確認済みの組み合わせ）
PINS = {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # 演習と同じく、Colab 同梱の torchao は peft 0.21 と合わないので外す
%pip install -q {spec}

import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定と Drive への接続

このノートブックは Drive にある学習データと評価の質問を読むので、Drive への接続が必須です。許可の画面では、演習と同じアカウントを選び、すべての項目にチェックを入れます。

**見るところ**: 設定の一覧が出ること。ファイルが見つからないというエラーが出たら、Drive の同期を少し待ってから、このセルを実行し直します。

In [ ]:
from pathlib import Path
import os, json, time, shutil
from google.colab import drive

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 10,   # 1 語に質問が 2 通りあるので、10 周で同じ説明文を 20 回見る（演習の 1 件 20 回と同じ）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,    # 説明文は最長 212 字。演習の 96 トークンでは途中で切れるおそれがある
    "gen": {"do_sample": False},   # 演習と同じ貪欲生成。学習前後で生成の条件をそろえる
    "work_dir": "/content/homework_glossary",
    "drive_dir": "/content/drive/MyDrive/aie2026/homework_glossary",   # 演習の qlora_lab とは分ける
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive")
DRIVE = Path(CONFIG["drive_dir"])
for name in ["glossary_sft.jsonl", "eval_questions.json"]:
    assert (DRIVE / name).exists(), f"{DRIVE / name} が見つかりません。Drive の同期を待ってから、このセルを実行し直してください"
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 学習データと評価の質問の読み込み

**やること**: 学習データ 116 件と、評価の質問 25 問を読みます。評価の質問は 3 区分です。

| 区分 | 意味 | 採点 |
| --- | --- | --- |
| `same` | 学習データと同じ言い回し（「{語}」とは何ですか？） | 説明文の鍵になる語句 2 つが答えに入っていれば ○ |
| `paraphrase` | 同じ語を、別の言い回しか別名で聞く（学習データには無い文） | 同上 |
| `unknown` | 用語集に無い語 | 「知らない」と答えたら ○、言い切ったら × |

**見るところ**: `学習データ: 116 件 | 評価の質問: {'same': 10, 'paraphrase': 10, 'unknown': 5}` と出ること。`same` の質問が学習データに無い、または `paraphrase` と `unknown` の質問が学習データにあると、ここでエラーになります。

In [ ]:
from collections import Counter

train_records = [json.loads(line) for line in (DRIVE / "glossary_sft.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
EVAL = json.loads((DRIVE / "eval_questions.json").read_text(encoding="utf-8"))
questions = EVAL["questions"]
REFUSAL = EVAL["refusal_phrases"]   # 「知らない」と答えたとみなす語句

train_questions = {r["messages"][0]["content"] for r in train_records}
for q in questions:
    seen = q["question"] in train_questions
    if q["category"] == "same":
        assert seen, f"学習と同じ言い回しのはずの質問が学習データにありません: {q['question']}"
    else:
        assert not seen, f"学習データにある質問が混ざっています: {q['question']}"
print("学習データ:", len(train_records), "件 | 評価の質問:", dict(Counter(q["category"] for q in questions)))
print("学習例の 1 件:", train_records[0]["messages"])

## 3. モデルの読み込みと学習前の評価

**やること**: 演習と同じく、量子化なし（32bit）でモデルを読み込みます。学習データと同じ形で、システムプロンプトを付けずに質問だけを渡し、25 問に答えさせます。

**見るところ**: 1 行の要約（区分ごとの ○ の数と秒数）と、25 問の表。学習前のモデルは、一般的な語（LoRA など）は自分の知識で答えられても、講座に固有の語は知りようがありません。用語集に無い語にどう答えるかも見ます。答えが長いので、評価には数分かかります。

`... is falling back to its reference PyTorch implementation ...` は演習と同じく無視してよい表示です。

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option("display.max_colwidth", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", round(model.get_memory_footprint() / 1e9, 2))

def generate(model, question: str) -> str:
    # 学習データと同じく、システムプロンプトなしで user の発言だけを渡す
    messages = [{"role": "user", "content": question}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluate(model, label: str):
    '''25 問を生成して採点し、区分ごとの要約と質問ごとの結果を返す。結果は Drive にもすぐ写す'''
    rows = []
    t0 = time.time()
    for q in questions:
        raw = generate(model, q["question"])
        row = {"id": q["id"], "category": q["category"], "kind": q["kind"], "term": q["term"], "question": q["question"], "raw": raw}
        if q["category"] in ("same", "paraphrase"):
            row["hit"] = [w for w in q["must_include"] if w in raw]
            row["ok"] = len(row["hit"]) == len(q["must_include"])
        else:
            row["refused"] = any(p in raw for p in REFUSAL)
            row["ok"] = row["refused"]   # 用語集に無い語は「知らない」と答えたら ○
        rows.append(row)
    summary = {"label": label, "seconds": round(time.time() - t0, 1)}
    for cat in ["same", "paraphrase", "unknown"]:
        sub = [r for r in rows if r["category"] == cat]
        summary[cat] = f"{sum(r['ok'] for r in sub)}/{len(sub)}"
    path = WORK / f"eval_glossary_{label}.json"
    path.write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    shutil.copy(path, DRIVE / path.name)   # セッションが切れても結果が残るように、すぐ Drive へ写す
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"区分": r["category"], "語": r["term"], "質問": r["question"], "○×": "○" if r["ok"] else "×", "出力": r["raw"]} for r in before_rows]))

## 4. LoRA 学習（10 エポック、150 ステップ）

**やること**: 演習と同じ LoRA の設定で、116 件を 10 周学習します。バッチ 4 × 勾配累積 2 ＝ 1 ステップ 8 件なので、1 周 15 ステップ、10 周で 150 ステップです。

**見るところ**

- `trainable%`: 演習と同じ設定なので、演習と同じ 0.84% 前後になるはずです。
- `Training Loss` の表（10 ステップごと、15 行）: どこまで下がるか。
- `学習時間: ... 秒 | ステップ数: 150 | エポック: 10.0`

**うまくいかないとき**: 途中で止まったら、出力の赤い文を控えてください。チェックポイントは 50 ステップごとに `/content/homework_glossary/ckpt/` に保存されます（Colab の一時領域なので、ランタイムを削除すると消えます）。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)   # 各行は {"messages": [user, assistant]}。演習の to_chat と同じ形
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=50, save_total_limit=2,
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
shutil.copytree(ADAPTER_DIR, DRIVE / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()   # 学習モードのままだと dropout が効き、評価の出力が揺れる

## 5. 学習後の評価（学習前と同じ 25 問、同じ生成設定）

**やること**: 学習後のモデルで、学習前と同じ 25 問を解かせ、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**: 区分ごとの ○ の数が、学習前からどう変わったか。とくに、言い回しを変えた質問（`paraphrase`）と、用語集に無い語（`unknown`）の答え方。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
display(pd.DataFrame([before_summary, after_summary]).set_index("label"))
display(pd.DataFrame([{"区分": b["category"], "語": b["term"], "質問": b["question"], "学習前": "○" if b["ok"] else "×", "学習後": "○" if a["ok"] else "×",
                       "学習前の出力": b["raw"], "学習後の出力": a["raw"]} for b, a in zip(before_rows, after_rows)]))

## 6. 結果の保存

**やること**: 設定、版、学習の数字、学習前後の要約を `result_glossary.json` にまとめ、評価の 2 ファイルと一緒に Drive の保存先へ写します。

**見るところ**: 最後に `Drive に保存しました: /content/drive/MyDrive/aie2026/homework_glossary` と出ること。終わったら、ランタイムを接続解除して削除します。

In [ ]:
loss_log = [{"step": x["step"], "loss": x["loss"]} for x in trainer.state.log_history if "loss" in x]
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）。講座の用語集の Q&A（116 件）で学習",
    "train_records": len(train_records),
    "train_seconds": TRAIN_SECONDS,
    "steps": trainer.state.global_step,
    "loss_log": loss_log,
    "before": before_summary, "after": after_summary,
    "executed_training_myself": True,
}
(WORK / "result_glossary.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
for name in ["result_glossary.json", "eval_glossary_before.json", "eval_glossary_after.json"]:
    shutil.copy(WORK / name, DRIVE / name)
print(json.dumps({k: v for k, v in result.items() if k != "loss_log"}, ensure_ascii=False, indent=2))
print("Drive に保存しました:", DRIVE)